# Ejercicio 1: features de fecha y hora

El mismo `viajes_taxi.csv` de la sesión pasada, sin los errores que la exploración confirmó (tarifa 999.99 y duración negativa).

Corre todas las celdas antes de tocar nada (**Run All**, en la barra de arriba del notebook): corre de principio a fin sin los `TODO` resueltos. Después resuelve cada `TODO` en su celda y vuelve a correr desde esa celda hacia abajo.

In [1]:
import numpy as np
import pandas as pd

viajes = pd.read_csv("viajes_taxi.csv")
viajes = viajes[(viajes["tarifa"] != 999.99) & (viajes["duracion_min"] >= 0)].copy()
inicio = pd.to_datetime(viajes["fecha_hora_inicio"])

## Paso 1: partes de la fecha

In [2]:
# TODO 1: la hora (0 a 23) y el dia de la semana (0 es lunes) de inicio.
viajes["hora"] = inicio.dt.hour
viajes["dia_semana"] = inicio.dt.dayofweek
# TODO 2: es_fin_de_semana, 1 los sabados y domingos, 0 el resto.
viajes["es_fin_de_semana"] = (viajes["dia_semana"] >= 5).astype(int)

## Paso 2: hora pico

In [3]:
horas_pico = [7, 8, 9, 16, 17, 18, 19]
# TODO 3: es_hora_pico, 1 si la hora esta en horas_pico y no es fin de
# semana, 0 el resto.
viajes["es_hora_pico"] = (viajes["hora"].isin(horas_pico) & (viajes["es_fin_de_semana"] == 0)).astype(int)

## Paso 3: la hora como ciclo

In [4]:
# TODO 4: hora_sin y hora_cos, la hora como un punto en un reloj de 24
# horas.
viajes["hora_sin"] = np.sin(2 * np.pi * viajes["hora"] / 24)
viajes["hora_cos"] = np.cos(2 * np.pi * viajes["hora"] / 24)

columnas = ["fecha_hora_inicio", "hora", "dia_semana", "es_fin_de_semana", "es_hora_pico", "hora_sin", "hora_cos"]
viajes[columnas].head()

,fecha_hora_inicio,hora,dia_semana,es_fin_de_semana,es_hora_pico,hora_sin,hora_cos
0,2026-09-06 13:46,13,6,1,0,-0.258819,-9.659258e-01
1,2026-10-01 22:22,22,3,0,0,-0.500000,8.660254e-01
2,2026-09-17 19:37,19,3,0,1,-0.965926,2.588190e-01
3,2026-10-04 18:09,18,6,1,0,-1.000000,-1.836970e-16
4,2026-10-12 09:28,9,0,0,1,0.707107,-7.071068e-01


## Paso 4: ¿la hora pico aporta algo a la tarifa?

In [5]:
normales = viajes[(viajes["tarifa"] != 70.0) & (viajes["distancia_km"] > 0)].copy()
normales["tarifa_por_km"] = normales["tarifa"] / normales["distancia_km"]
print("Tarifa por km, mediana, segun hora pico:")
print(normales.groupby("es_hora_pico")["tarifa_por_km"].median().round(2))

Tarifa por km, mediana, segun hora pico:
es_hora_pico
0    3.22
1    3.52
Name: tarifa_por_km, dtype: float64


## Paso 5: qué tan lejos quedan dos horas

In [6]:
def distancia(fila_a, fila_b):
    a = np.array([fila_a["hora_sin"], fila_a["hora_cos"]])
    b = np.array([fila_b["hora_sin"], fila_b["hora_cos"]])
    return np.sqrt(((a - b) ** 2).sum())


ejemplo = viajes.drop_duplicates("hora").set_index("hora")
for h1, h2 in [(23, 0), (11, 12)]:
    if h1 in ejemplo.index and h2 in ejemplo.index:
        print(f"{h1} y {h2}: como numero {abs(h1 - h2)}, como ciclo {distancia(ejemplo.loc[h1], ejemplo.loc[h2]):.2f}")
    else:
        print(f"{h1} y {h2}: todavia no hay horas distintas para comparar")

23 y 0: como numero 23, como ciclo 0.26
11 y 12: como numero 1, como ciclo 0.26
